In [ ]:
import time as time_module
from datetime import date, datetime, timedelta, time
from zoneinfo import ZoneInfo

import pandas as pd
import requests


# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

TABLE_NAME = "dim_crypto_rates_weekly"

PERIOD_STARTS = [
    "2026-01-05",
    "2026-01-12",
    "2026-01-19",
    "2026-01-26",
]

CRYPTOCURRENCIES = [
    "BTC",
    "ETH",
    "SOL",
    "BNB",
    "XRP",
    "USDT",
    "USDT.TRC20",
    "USDC",
]

REPORTING_CURRENCY = "EUR"

LOCAL_TIMEZONE = ZoneInfo("Europe/Riga")
UTC_TIMEZONE = ZoneInfo("UTC")

BINANCE_API_URL = "https://api.binance.com/api/v3/klines"

REQUEST_TIMEOUT_SECONDS = 20
REQUEST_RETRIES = 3
REQUEST_DELAY_SECONDS = 0.2

TRUNCATE_BEFORE_LOAD = True


# ------------------------------------------------------------------
# Currency normalization
# ------------------------------------------------------------------

def normalize_crypto(currency: str) -> str:
    """
    Converts network-specific stablecoin names into a common currency code.
    """
    aliases = {
        "USDT.TRC20": "USDT",
        "USDT.ERC20": "USDT",
        "USDT.BEP20": "USDT",
    }

    normalized = currency.strip().upper()
    return aliases.get(normalized, normalized)


# ------------------------------------------------------------------
# Date and timestamp helpers
# ------------------------------------------------------------------

def datetime_to_milliseconds(value: datetime) -> int:
    return int(value.timestamp() * 1000)


def parse_period_starts(periods: list[str]) -> list[date]:
    return [
        datetime.strptime(period, "%Y-%m-%d").date()
        for period in periods
    ]


def build_periods(
    period_starts: list[str],
) -> list[tuple[date, date, datetime]]:
    """
    Builds weekly periods.

    Each target timestamp represents 23:59 in the configured local
    timezone on the final day of the period.
    """
    start_dates = parse_period_starts(period_starts)
    result = []

    for index, start_date in enumerate(start_dates):
        if index + 1 < len(start_dates):
            end_date = start_dates[index + 1] - timedelta(days=1)
        else:
            end_date = start_date + timedelta(days=6)

        target_local = datetime.combine(
            end_date,
            time(hour=23, minute=59),
            tzinfo=LOCAL_TIMEZONE,
        )

        result.append(
            (
                start_date,
                end_date,
                target_local,
            )
        )

    return result


# ------------------------------------------------------------------
# Binance API
# ------------------------------------------------------------------

def request_klines(
    symbol: str,
    target_datetime_utc: datetime,
) -> list:
    """
    Requests one-minute candles within a one-hour window around
    the target timestamp.
    """
    window_start = target_datetime_utc - timedelta(minutes=30)
    window_end = target_datetime_utc + timedelta(minutes=30)

    params = {
        "symbol": symbol,
        "interval": "1m",
        "startTime": datetime_to_milliseconds(window_start),
        "endTime": datetime_to_milliseconds(window_end),
        "limit": 1000,
    }

    for attempt in range(1, REQUEST_RETRIES + 1):
        try:
            response = requests.get(
                BINANCE_API_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )

            response.raise_for_status()

            payload = response.json()

            if isinstance(payload, list):
                return payload

            print(
                f"Unexpected API response for {symbol}: "
                f"{type(payload).__name__}"
            )

            return []

        except requests.RequestException as error:
            print(
                f"Request failed for {symbol}. "
                f"Attempt {attempt}/{REQUEST_RETRIES}: {error}"
            )

            if attempt < REQUEST_RETRIES:
                time_module.sleep(attempt)

    return []


def get_nearest_close_price(
    symbol: str,
    target_datetime_utc: datetime,
) -> tuple[float | None, datetime | None]:
    """
    Returns the close price from the candle nearest to the requested
    target timestamp.
    """
    candles = request_klines(
        symbol=symbol,
        target_datetime_utc=target_datetime_utc,
    )

    if not candles:
        return None, None

    target_milliseconds = datetime_to_milliseconds(
        target_datetime_utc
    )

    nearest_candle = min(
        candles,
        key=lambda candle: abs(
            int(candle[0]) - target_milliseconds
        ),
    )

    candle_datetime_utc = datetime.fromtimestamp(
        int(nearest_candle[0]) / 1000,
        tz=UTC_TIMEZONE,
    )

    candle_datetime_local = candle_datetime_utc.astimezone(
        LOCAL_TIMEZONE
    )

    close_price = float(nearest_candle[4])

    return close_price, candle_datetime_local


# ------------------------------------------------------------------
# Exchange-rate calculation
# ------------------------------------------------------------------

def get_reporting_currency_stablecoin_rate(
    target_datetime_utc: datetime,
) -> tuple[float | None, datetime | None]:
    """
    Returns the market price of the reporting currency against USDT.

    Example:
        EURUSDT = number of USDT per EUR.
    """
    symbol = f"{REPORTING_CURRENCY}USDT"

    return get_nearest_close_price(
        symbol=symbol,
        target_datetime_utc=target_datetime_utc,
    )


def get_crypto_reporting_rate(
    currency: str,
    target_datetime_utc: datetime,
) -> tuple[float | None, datetime | None, str]:
    """
    Returns the value of one cryptocurrency unit in the reporting
    currency.

    Resolution order:

    1. Reporting currency itself.
    2. Stablecoin conversion through REPORTING_CURRENCY/USDT.
    3. Direct CRYPTO/REPORTING_CURRENCY market.
    4. Indirect conversion through CRYPTO/USDT.
    """
    normalized_currency = normalize_crypto(currency)

    if normalized_currency == REPORTING_CURRENCY:
        return 1.0, None, REPORTING_CURRENCY

    reporting_usdt_rate, reporting_usdt_time = (
        get_reporting_currency_stablecoin_rate(
            target_datetime_utc
        )
    )

    if reporting_usdt_rate is None:
        return None, None, f"NO_{REPORTING_CURRENCY}USDT"

    if normalized_currency in {"USDT", "USDC"}:
        rate = 1 / reporting_usdt_rate

        return (
            rate,
            reporting_usdt_time,
            f"{REPORTING_CURRENCY}USDT",
        )

    direct_symbol = (
        f"{normalized_currency}{REPORTING_CURRENCY}"
    )

    direct_price, direct_time = get_nearest_close_price(
        symbol=direct_symbol,
        target_datetime_utc=target_datetime_utc,
    )

    if direct_price is not None:
        return direct_price, direct_time, direct_symbol

    stablecoin_symbol = f"{normalized_currency}USDT"

    stablecoin_price, stablecoin_time = (
        get_nearest_close_price(
            symbol=stablecoin_symbol,
            target_datetime_utc=target_datetime_utc,
        )
    )

    if stablecoin_price is None:
        return None, None, stablecoin_symbol

    reporting_rate = (
        stablecoin_price / reporting_usdt_rate
    )

    source = (
        f"{stablecoin_symbol} / "
        f"{REPORTING_CURRENCY}USDT"
    )

    return reporting_rate, stablecoin_time, source


# ------------------------------------------------------------------
# Dataset creation
# ------------------------------------------------------------------

def collect_weekly_rates() -> list[dict]:
    rows = []

    created_at = datetime.now(
        LOCAL_TIMEZONE
    ).replace(tzinfo=None)

    for period_start, period_end, target_local in build_periods(
        PERIOD_STARTS
    ):
        target_utc = target_local.astimezone(
            UTC_TIMEZONE
        )

        print("=" * 70)
        print(f"Period: {period_start} - {period_end}")
        print(
            "Target time: "
            f"{target_local:%Y-%m-%d %H:%M:%S %Z}"
        )
        print("=" * 70)

        for currency in CRYPTOCURRENCIES:
            normalized_currency = normalize_crypto(
                currency
            )

            rate, candle_time, source = (
                get_crypto_reporting_rate(
                    currency=currency,
                    target_datetime_utc=target_utc,
                )
            )

            if rate is None:
                print(
                    f"{currency}: rate not found "
                    f"| source={source}"
                )
                continue

            effective_rate_time = (
                candle_time or target_local
            )

            rows.append(
                {
                    "period_start": period_start,
                    "period_end": period_end,
                    "target_time": (
                        target_local.replace(tzinfo=None)
                    ),
                    "rate_time": (
                        effective_rate_time.replace(
                            tzinfo=None
                        )
                    ),
                    "currency": currency,
                    "normalized_currency": (
                        normalized_currency
                    ),
                    "rate": float(rate),
                    "source": source,
                    "created_at": created_at,
                }
            )

            print(
                f"{currency}: {rate:.12f} "
                f"| source={source} "
                f"| candle={effective_rate_time:%Y-%m-%d %H:%M:%S %Z}"
            )

            time_module.sleep(
                REQUEST_DELAY_SECONDS
            )

        print()

    return rows


# ------------------------------------------------------------------
# Delta Lake write
# ------------------------------------------------------------------

def write_rates_to_lakehouse(rows: list[dict]) -> None:
    if not rows:
        print("No rows available for insertion.")
        return

    if TRUNCATE_BEFORE_LOAD:
        spark.sql(
            f"TRUNCATE TABLE {TABLE_NAME}"
        )

    pandas_df = pd.DataFrame(rows)

    spark_df = spark.createDataFrame(
        pandas_df
    )

    (
        spark_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLE_NAME)
    )

    print(f"Inserted rows: {len(rows)}")
    print(f"Target table: {TABLE_NAME}")


def main() -> None:
    rows = collect_weekly_rates()
    write_rates_to_lakehouse(rows)


main()